Scenario: The Field Marketing Lead has asked you to help prepare their team for conference season. You create a Conference Prep Agent that uses 3 tools: web search to collect relevant news and search for up to date information, Wikipedia to provide company history and details, and the team's internal notes and artifacts.

## Install  BeeAI Framework and set up our environment.



In [ ]:
%pip install -Uqq uv
!uv pip install --system -q arize-phoenix arize-phoenix-otel s3fs fsspec==2025.3.0 \
 gcsfs nest_asyncio openinference-instrumentation-beeai jedi \
 beeai-framework beeai-framework[duckduckgo] beeai-framework[llama_index] beeai-framework[wikipedia] \
 langchain langchain_community wikipedia unstructured requests==2.32.4

![ -d beeai_framework_synthetic_data ] || git clone https://github.com/sandijean90/beeai_framework_synthetic_data.git

from IPython.display import HTML, display
def set_css(*args, **kwargs):
    display(HTML("\n<style>\n pre{\n white-space: pre-wrap;\n}\n</style>\n"))
get_ipython().events.register("pre_run_cell", set_css)

In [ ]:
%pip install -Uqq arize-phoenix arize-phoenix-otel s3fs fsspec==2025.3.0 \
 gcsfs nest_asyncio openinference-instrumentation-beeai jedi \
 beeai-framework beeai-framework[duckduckgo] beeai-framework[llama_index] beeai-framework[wikipedia] \
 langchain langchain_community wikipedia unstructured requests==2.32.4

!git clone https://github.com/sandijean90/beeai_framework_synthetic_data.git

from IPython.display import HTML, display
def set_css():
    display(HTML("\n<style>\n pre{\n white-space: pre-wrap;\n}\n</style>\n"))
get_ipython().events.register("pre_run_cell", set_css)

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.2/262.2 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━

Import modules:


In [ ]:
import os
from datetime import date

import phoenix as px
from phoenix.otel import register
from dotenv import load_dotenv

from beeai_framework.agents.experimental import RequirementAgent
from beeai_framework.agents.experimental.requirements.conditional import ConditionalRequirement
from beeai_framework.backend import ChatModel, ChatModelParameters
from beeai_framework.backend.document_loader import DocumentLoader
from beeai_framework.backend.embedding import EmbeddingModel
from beeai_framework.backend.text_splitter import TextSplitter
from beeai_framework.backend.vector_store import VectorStore
from beeai_framework.memory import SummarizeMemory
from beeai_framework.middleware.trajectory import GlobalTrajectoryMiddleware
from beeai_framework.tools import Tool, tool
from beeai_framework.tools.search.duckduckgo import DuckDuckGoSearchTool
from beeai_framework.tools.search.retrieval import VectorStoreSearchTool
from beeai_framework.tools.search.wikipedia import WikipediaTool, WikipediaToolInput
from beeai_framework.tools.think import ThinkTool
from openinference.instrumentation.beeai import BeeAIInstrumentor

### Choose LLM (Ollama)

In [ ]:
provider = "ollama"
model = "granite3.3"

if provider == "ollama":
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    !nohup ollama serve >/dev/null 2>&1 &
    !ollama pull $model

llm = ChatModel.from_name(f"{provider}:{model}", ChatModelParameters(temperature=1))

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



### System Prompt

In [ ]:
instruct_prompt = """You help field marketing teams prep for conferences by answering questions on companies that they need to prepare to talk to. You produce quick and actionable briefs, doing your best to answer the user's question.

Today's date is {todays_date}.

Tools:
- ThinkTool: Helps you plan and reason before you act. Use this tool when you need to think.
- DuckDuckGoSearchTool: Use this tool to collect current information on agendas, speakers, news, competitor moves. Include title + date + link to the resources you find in your answer. Do not use this tool for internal notes or artifacts.
- wikipedia_tool: Use this tool to get company/org history (not for breaking news). Only look up company names as the input.
- internal_document_search: past meetings, playbooks, artifacts. If you use information from this in your response, label it as [Internal]. Always use this tool when internal notes or content are referenced.

Basic Rules:
- Be concise and practical.
- Favor recent info (agenda/news ≤30 days; exec changes/funding ≤180 days); flag older items.
- If you don't know, say so. Don't make things up.
"""

In [ ]:
todays_date = date.today().strftime("%B %d, %Y")
formatted_instruct_prompt = instruct_prompt.format(
        todays_date=todays_date)

Memory:

In [ ]:
memory = SummarizeMemory(llm)

### Tools

In [ ]:
think_tool = ThinkTool()

The **DuckDuckGoSearchTool** is a Web Search tool that provides relevant data from the internet to the LLM


In [ ]:
search_tool = DuckDuckGoSearchTool()

In [ ]:
@tool
async def wikipedia_tool(query: str) -> str:
    """Search Wikipedia for factual and historical information.

    Args:
        query: The topic or company name to look up.

    Returns:
        Text content from the matching Wikipedia article.
    """
    wiki = WikipediaTool(language="en")
    response = await wiki.run(WikipediaToolInput(query=query, full_text=False))
    return response.get_text_content()

### RAG

In [ ]:
!ollama pull nomic-embed-text:latest
embedding_model = EmbeddingModel.from_name("ollama:nomic-embed-text")

In [ ]:
# Read the synthetic data from the public github repo
with open('/content/beeai_framework_synthetic_data/rag_conference_prep_agent.txt', 'r') as file:
    content = file.read()
print(content)

==== RAG DOCUMENT ====
Account ID: A001
Type: call_note
Title: Shopify: Observability & Personalization discussion (Jun 18, 2025)
Created At: 2025-06-18T15:40:00Z
Author: Jordan Blake
Conference ID: None
Tags: observability, alert-fatigue, security, ml, personalization, roi

Content:
Context:
Shopify's platform teams are revisiting observability to reduce alert fatigue while the
Growth org explores AI-driven merchandising. Attendees included VP Eng Platforms (Marina L.),
Security Architect (Sofia N.).

Current stack & pain points:
- Mixed cloud (GCP + AWS), Kubernetes,
Kafka, Prometheus, Grafana, homegrown alert rules. 
- Noisy alerts with low precision; teams mute
channels during oncall rotations. 
- Limited root-cause hints; SREs manually correlate metrics,
logs, and traces. 
- Security requires tighter access reviews and clearer data residency posture for
any ML features.

Proposed approach:
- Introduce ML anomaly scoring layer that consumes metrics and
key business KPIs (checkout s

In [ ]:
loader = DocumentLoader.from_name(
    name="langchain:UnstructuredMarkdownLoader", file_path="/content/beeai_framework_synthetic_data/rag_conference_prep_agent.txt"
)
try:
    documents = await loader.load()
except Exception as e:
    print(f"Failed to load documents: {e}")

# Split documents into chunks
text_splitter = TextSplitter.from_name(
    name="langchain:RecursiveCharacterTextSplitter", chunk_size=1000, chunk_overlap=200
)
documents = await text_splitter.split_documents(documents)
print(f"Loaded {len(documents)} document chunks")

Loaded 14 document chunks


In [ ]:
# Create vector store and add documents
vector_store = VectorStore.from_name(name="beeai:TemporalVectorStore", embedding_model=embedding_model)
await vector_store.add_documents(documents=documents)
print("Vector store populated with documents")


Vector store populated with documents


In [ ]:
# Create the vector store search tool
internal_document_search = VectorStoreSearchTool(vector_store=vector_store)

### Control Agent Behavior

In [ ]:
requirement_1 = ConditionalRequirement(ThinkTool, consecutive_allowed=False, force_at_step=1 )

requirement_2 = ConditionalRequirement(wikipedia_tool, only_after=ThinkTool, consecutive_allowed=True, priority=10,)

requirement_3 = ConditionalRequirement(DuckDuckGoSearchTool, only_after=ThinkTool, consecutive_allowed=True, min_invocations=1, max_invocations=3 ,priority=15,)

requirement_4 = ConditionalRequirement(internal_document_search, only_after=ThinkTool, consecutive_allowed=True, min_invocations=1, priority=20,)


In [ ]:
load_dotenv()

px_session = px.launch_app()

tracer_provider = register(
    project_name="conference-prep-agent",
    endpoint="http://localhost:6006/v1/traces",
)
BeeAIInstrumentor().instrument(tracer_provider=tracer_provider)

In [ ]:
print(f"Phoenix UI: {px_session.url}")

##  Assemble Agent

In [ ]:
agent = RequirementAgent(
    llm=llm,
    instructions=formatted_instruct_prompt,
    memory=memory,
    tools=[ThinkTool(), DuckDuckGoSearchTool(), wikipedia_tool, internal_document_search],
    requirements=[requirement_1, requirement_2, requirement_3, requirement_4],
    middlewares=[GlobalTrajectoryMiddleware(included=[Tool])],
)

In [ ]:
question = "I'm planning on meeting the Moderna rep at the next conference. Give me a one pager and remind me where we left off on previous internal discussions."

### Run Agent

In [ ]:
response = await agent.run(question, max_retries_per_step=3, total_max_retries=25)
print(response.last_message.text)

--> 🛠️ ThinkTool[think][start]: {"input": {"input": {"thoughts": "First, I will gather any relevant information about Moderna from recent sources using DuckDuckGo. For the internal discussion summary, I'll search through our past meetings and playbooks with the internal document search tool.", "next_step": ["Search recent news about Moderna to understand their latest updates or developments", "Review any unfinished notes from prior discussions within our company"]}}}
<-- 🛠️ ThinkTool[think][finish]: "OK"
--> 🛠️ DuckDuckGoSearchTool[DuckDuckGo][start]: {"input": {"input": {"query": "Moderna latest updates 2025"}}}
<-- 🛠️ DuckDuckGoSearchTool[DuckDuckGo][finish]: [{"title": "Moderna's 2025-2026 COVID-19 Vaccines Get FDA Approval", "description": "The Food and Drug Administration (FDA) has approved the 2025 -2026 formulations of Moderna's COVID-19 vaccines Spikevax ® and mNexspike ®. The updated mRNA vaccines target the LP.8.1 variant of SARS-CoV-2, which was selected as the preferred str

In [ ]:
px_session.view()

📺 Opening a view to the Phoenix app. The app is running at https://olsktqy1xk2-496ff2e9c6d22116-6006-colab.googleusercontent.com/
